# Readiness check

Run this notebook before the workshop. It verifies the reference-data
environment and includes an Earth Engine project check for participants
who plan to analyze a selected watershed.

**Success means:** `cnkit 1.1.0`, a checksum-verified data bundle or a
local workshop folder, and three known runoff values. If you enable the
Earth Engine check, success also prints `42`.


## Step 1 — Establish the reproducible environment

The setup cell performs infrastructure work only. It does not calculate
a curve number and it does not authenticate Earth Engine.

1. It looks for an existing workshop folder.
2. If needed, it downloads the versioned data bundle and verifies its
   SHA-256 checksum before extraction.
3. It loads `cnkit` 1.1.0 from the installed package, the workshop's
   portable module, or PyPI—in that order.
4. It defines `DATA_DIR` and `PREPARED_DIR` so every later code cell
   records where its inputs came from.

The printed version and source lines are part of the analytical
provenance. Retain them when exporting a notebook for review.


In [ ]:
# V3 portable setup: local repository, GitHub Pages bundle, or Colab.
from pathlib import Path
import hashlib
import importlib
import importlib.util
import os
import subprocess
import sys
import urllib.request
import zipfile

CNKIT_VERSION = "1.1.0"
BUNDLE_URL = (
    "https://skp703.github.io/cn-workshop-2026/"
    "downloads/cn_workshop_v3_data.zip"
)
BUNDLE_SHA256 = "925861246fe9520c4b7f399227ca6133e60f27a5063a73c9fb6715cd9904780c"


def _is_workshop_root(path):
    return (path / "data" / "sites.csv").exists() and (path / "prepared").exists()


def _find_workshop_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content/cnkit_workshop"),
    ]
    requested = os.environ.get("CNKIT_WORKSHOP_HOME")
    if requested:
        candidates.insert(0, Path(requested).expanduser())
    for candidate in candidates:
        candidate = candidate.resolve()
        if _is_workshop_root(candidate):
            return candidate, "existing workshop folder"

    destination = Path("/content/cnkit_workshop") if Path("/content").exists() else Path.cwd() / ".cnkit_workshop"
    destination.mkdir(parents=True, exist_ok=True)
    archive = destination / "cn_workshop_v3_data.zip"
    print("Downloading the versioned V3 workshop bundle...")
    request = urllib.request.Request(BUNDLE_URL, headers={"User-Agent": "cn-workshop-v3"})
    with urllib.request.urlopen(request, timeout=120) as response, archive.open("wb") as handle:
        handle.write(response.read())
    digest = hashlib.sha256(archive.read_bytes()).hexdigest()
    if digest != BUNDLE_SHA256:
        raise RuntimeError(
            "Workshop bundle checksum mismatch. Expected %s, received %s. "
            "Delete %s and try again." % (BUNDLE_SHA256, digest, archive)
        )
    with zipfile.ZipFile(archive) as zipped:
        zipped.extractall(destination)
    if not _is_workshop_root(destination):
        raise RuntimeError("The workshop bundle downloaded but required files are missing.")
    return destination.resolve(), "checksum-verified workshop download"


WORKSHOP_ROOT, DATA_SOURCE = _find_workshop_root()
DATA_DIR = WORKSHOP_ROOT / "data"
PREPARED_DIR = WORKSHOP_ROOT / "prepared"


def _load_cnkit():
    try:
        import cnkit as package
        if getattr(package, "__version__", None) == CNKIT_VERSION:
            return package, "installed package"
    except ImportError:
        pass

    for candidate in [
        WORKSHOP_ROOT / "vendor" / "cnkit.py",
        WORKSHOP_ROOT / "cnkit.py",
        Path.cwd() / "vendor" / "cnkit.py",
        Path.cwd().parent / "vendor" / "cnkit.py",
    ]:
        if candidate.exists():
            spec = importlib.util.spec_from_file_location("cnkit", candidate)
            package = importlib.util.module_from_spec(spec)
            sys.modules["cnkit"] = package
            spec.loader.exec_module(package)
            return package, str(candidate)

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "cnkit==" + CNKIT_VERSION]
    )
    importlib.invalidate_caches()
    import cnkit as package
    return package, "PyPI"


def activate_full_cnkit():
    """Return the installed package with data, delineation, and GEE modules."""
    global cnkit, CNKIT_SOURCE
    if hasattr(cnkit, "__path__"):
        return cnkit
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "cnkit[gee]==" + CNKIT_VERSION]
    )
    for name in [key for key in sys.modules if key == "cnkit" or key.startswith("cnkit.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
    cnkit = importlib.import_module("cnkit")
    CNKIT_SOURCE = "PyPI with Earth Engine dependencies"
    return cnkit


cnkit, CNKIT_SOURCE = _load_cnkit()
print("cnkit version:", getattr(cnkit, "__version__", CNKIT_VERSION + " workshop module"))
print("cnkit source :", CNKIT_SOURCE)
print("data source  :", DATA_SOURCE)
print("data folder  :", DATA_DIR)
print("setup complete")


## Step 2 — Identify the library layers

| Layer | Responsibility |
|---|---|
| `cnkit.core` | Rainfall–runoff equation, inverse equation, and compositing mathematics |
| `cnkit.lookup` | NLCD–soil-group lookup tables, hydrologic condition, and unmapped-area accounting |
| `cnkit.delineate` | Outlet snapping, watershed delineation, area checks, geometry, and boundary provenance |
| `cnkit.gee` | Earth Engine requests for land cover, imperviousness, soils, and their pixelwise joint distribution |
| `cnkit.workflows` | Reproducible orchestration across years; it delegates the hydrologic calculations to the layers above |

The laboratory notebooks call these layers separately before showing
the corresponding convenience workflow. This makes the scientific
assumptions visible rather than embedding them in a single function.


## Step 3 — Verify the core calculation and data files

The numerical check calls `composite_runoff` for a known heterogeneous
watershed example. Internally, the function validates the curve numbers
and area weights, normalizes the weights, evaluates runoff on each
subarea, and also evaluates runoff from area-weighted CN and
area-weighted retention. The three expected values therefore test the
core equation and both aggregation pathways.

The second check confirms that the event, land-cover, soil, boundary,
and recorded Earth Engine products used later are present.


In [ ]:
from cnkit import composite_runoff

expected = [0.4745, 0.0949, 0.0277]
actual = [round(x, 4) for x in composite_runoff(1.0, [98, 55], [0.6, 0.4])]
print("weighting check:", actual)
assert actual == expected

required = [
    DATA_DIR / "events_01646000.csv",
    DATA_DIR / "landcover_streamcat.csv",
    DATA_DIR / "soils_hsg.csv",
    PREPARED_DIR / "difficult_run_gee_summary.json",
    PREPARED_DIR / "difficult_run_gee_trajectory.csv",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing workshop files:\n" + "\n".join(missing))
print("reference-data pathway: ready")


## Step 4 — Verify an Earth Engine project

Set `TEST_EARTH_ENGINE = True` after registering a Cloud project for
Earth Engine. Leave it at `False` when using the workshop reference data.

The project ID is requested at runtime and is not saved in this notebook.


In [ ]:
TEST_EARTH_ENGINE = False

if TEST_EARTH_ENGINE:
    import os
    from getpass import getpass
    import ee

    project = os.environ.get("CNKIT_EE_PROJECT") or getpass("Earth Engine project ID: ")
    ee.Authenticate()
    ee.Initialize(project=project)
    answer = ee.Number(21).multiply(2).getInfo()
    assert answer == 42
    print("Earth Engine check:", answer)
    print("live path: ready")
else:
    print("Reference-data environment: ready")
    print("Set TEST_EARTH_ENGINE=True to verify an Earth Engine project.")


## Step 5 — Select a watershed identifier

The live path accepts either a USGS gage number or outlet latitude and
longitude for a CONUS watershed. Difficult Run and Accotink Creek are
also available as fully documented reference analyses.
